# LeetCode 2119 — A Number After a Double Reversal bằng Python

Link đề: https://leetcode.com/problems/a-number-after-a-double-reversal/ — Độ khó: **Easy**

Skeleton LeetCode:

```python
class Solution:
    def isSameAfterReversals(self, num: int) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu phép **đảo ngược số** và vì sao chữ số 0 ở cuối "biến mất" sau khi đảo.
2. Biết đảo số bằng chuỗi (`str(n)[::-1]`) và bằng toán học (`% 10`, `// 10`).
3. Viết 4 cách: mô phỏng bằng chuỗi, mô phỏng bằng toán, quy tắc `num % 10`, và quy tắc trên chuỗi.
4. Rút ra insight: bài này thực chất chỉ hỏi "**số có tận cùng bằng 0 (và khác 0) hay không**".
5. Biết cách **chứng minh** một quy tắc rồi kiểm chứng bằng brute force trên mọi `num` trong giới hạn.

## 2. Đề bài & ví dụ

**Đảo ngược** một số nguyên là viết các chữ số của nó theo thứ tự ngược lại; số 0 đứng đầu bị bỏ (ví dụ `1200` → `0021` = `21`).

Cho số nguyên `num`. Đảo ngược `num` được `reversed1`, rồi đảo ngược `reversed1` được `reversed2`. Trả về `True` nếu `reversed2 == num`, ngược lại `False`.

| num | reversed1 | reversed2 | kết quả |
|---|---|---|---|
| `526` | `625` | `526` | `True` |
| `1800` | `81` | `18` | `False` |
| `0` | `0` | `0` | `True` |

Constraints: `0 <= num <= 10^6`.

In [1]:
examples = [(526, True), (1800, False), (0, True)]
for num, expected in examples:
    r1 = int(str(num)[::-1])
    r2 = int(str(r1)[::-1])
    print(f"num={num}, reversed1={r1}, reversed2={r2}, mong đợi={expected}")

num=526, reversed1=625, reversed2=526, mong đợi=True
num=1800, reversed1=81, reversed2=18, mong đợi=False
num=0, reversed1=0, reversed2=0, mong đợi=True


## 3. Kiến thức nền

### 3.1. Chuyển số ↔ chuỗi: `str()` và `int()`

`str(526)` là `"526"`; `int("625")` là `625`. Khi `int` đọc chuỗi có số 0 đứng đầu như `"0021"`, kết quả là `21` — **số 0 đứng đầu tự biến mất**. Đây chính là cơ chế của đề.

In [2]:
print(str(526), type(str(526)))
print(int("625"), int("0021"), int("000"))

526 <class 'str'>
625 21 0


### 3.2. Slicing đảo ngược `s[::-1]`

`s[::-1]` là chuỗi đảo ngược của `s` (bước nhảy `-1` đi từ cuối về đầu).

In [3]:
s = "1800"
print(s[::-1], int(s[::-1]))

0081 81


### 3.3. Toán tử `%` và `//` cho từng chữ số

- `n % 10` là **chữ số cuối** của `n` (ví dụ `1800 % 10 = 0`).
- `n // 10` là `n` sau khi **bỏ chữ số cuối** (ví dụ `1800 // 10 = 180`).

Hai phép này cho phép đảo số mà không cần chuỗi: lấy chữ số cuối rồi gắn vào đuôi kết quả bằng `rev = rev * 10 + digit`.

In [4]:
n = 1800
print(n % 10, n // 10)
n = 526
print(n % 10, n // 10, (n // 10) % 10, n // 100)

0 180
6 52 2 5


### 3.4. Hàm `endswith` và so sánh chuỗi

`s.endswith("0")` kiểm tra chuỗi kết thúc bằng `"0"`, còn `s[-1] == "0"` cho kết quả như vậy. Với số: `n % 10 == 0` nghĩa là chữ số cuối là `0`.

In [5]:
print("1800".endswith("0"), "526".endswith("0"), "1800"[-1] == "0")
print(1800 % 10 == 0, 526 % 10 == 0)

True False True
True False


### 3.5. Vòng `while` và "tách từng chữ số"

Mẫu quen thuộc để duyệt chữ số của một số nguyên dương `n` từ **phải sang trái**:

```python
while n > 0:
    digit = n % 10   # chữ số cuối
    n //= 10         # bỏ chữ số cuối
```

Vòng lặp dừng khi `n` về `0`. Với `n = 0` ngay từ đầu, thân vòng lặp **không chạy lần nào** — đây là lý do `reverse_math(0)` trả về `0` mà không cần xử lý riêng.

In [6]:
n = 4096
digits = []
while n > 0:
    digits.append(n % 10)
    n //= 10
print("các chữ số từ phải sang trái:", digits)
n = 0
count = 0
while n > 0:
    count += 1
print("n = 0 -> số vòng lặp =", count)

các chữ số từ phải sang trái: [6, 9, 0, 4]
n = 0 -> số vòng lặp = 0


## 4. Ý tưởng nền tảng — tính bằng tay

- `526` → đảo `625` → đảo `526`. Giống ban đầu.
- `1800` → đảo `0081` = `81` → đảo `18`. **Mất hai số 0** nên khác `1800`.
- `0` → đảo `0` → đảo `0`. Giống.

Vậy khi nào mất thông tin? Chỉ khi `num` có số 0 ở **cuối** (nó thành số 0 ở **đầu** sau lần đảo thứ nhất rồi bị bỏ). Số 0 ở giữa thì vô hại: `1005` → đảo `5001` → đảo `1005`, giữ nguyên. Còn `1050` có số 0 ở cuối nên vẫn mất: `1050` → `0501 = 501` → `105`.

In [7]:
for num in [526, 1800, 1005, 1050, 10, 7, 0]:
    r2 = int(str(int(str(num)[::-1]))[::-1])
    print(f"{num:5d} -> {r2:5d}  giống nhau? {r2 == num}  | chữ số cuối là 0? {num % 10 == 0}")

  526 ->   526  giống nhau? True  | chữ số cuối là 0? False
 1800 ->    18  giống nhau? False  | chữ số cuối là 0? True
 1005 ->  1005  giống nhau? True  | chữ số cuối là 0? False
 1050 ->   105  giống nhau? False  | chữ số cuối là 0? True
   10 ->     1  giống nhau? False  | chữ số cuối là 0? True
    7 ->     7  giống nhau? True  | chữ số cuối là 0? False
    0 ->     0  giống nhau? True  | chữ số cuối là 0? True


## 5. Cách 1 — Brute Force: làm đúng theo đề bằng chuỗi

Viết hàm `reverse_str(n)` rồi gọi hai lần.

In [8]:
def reverse_str(n):
    return int(str(n)[::-1])

def same_simulate_str(num):
    return reverse_str(reverse_str(num)) == num

for num, expected in examples:
    print(num, same_simulate_str(num), expected)

526 True True
1800 False False
0 True True


Độ phức tạp: với `d = số chữ số` của `num` (≤ 7), mỗi lần đảo `O(d)` → **Time `O(d)`**, **Space `O(d)`**. Nút thắt: tạo chuỗi và số trung gian để trả lời một câu hỏi *có/không* mà đáp án thật ra chỉ phụ thuộc vào một chữ số.

## 6. Ý tưởng thuật toán / bước đột phá

Gọi chữ số của `num` từ trái sang phải là `d₁ d₂ ... dₖ`.

- Nếu `dₖ ≠ 0`: lần đảo 1 cho `dₖ ... d₁` (không có 0 ở đầu vì `dₖ ≠ 0`, không mất chữ số). Lần đảo 2 trả lại `d₁ ... dₖ`. → `True`.
- Nếu `dₖ = 0` và `num ≠ 0`: lần đảo 1 cho chuỗi bắt đầu bằng `0`, bỏ 0 đó → mất ít nhất một chữ số → kết quả cuối **ít chữ số hơn** `num` → `False`.
- Ngoại lệ `num = 0`: đảo `0` vẫn là `0` → `True`.

Quy tắc cuối: **`num == 0` hoặc `num % 10 != 0`**.

## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng bằng toán (không dùng chuỗi)

Đảo số: `rev = 0; while n > 0: rev = rev*10 + n%10; n //= 10`. Gọi hai lần rồi so sánh.

In [9]:
def reverse_math(n):
    rev = 0
    while n > 0:
        rev = rev * 10 + n % 10
        n //= 10
    return rev

def same_simulate_math(num):
    return reverse_math(reverse_math(num)) == num

for num, expected in examples:
    print(num, same_simulate_math(num), expected)

526 True True
1800 False False
0 True True


In [10]:
class SolutionMath:
    def isSameAfterReversals(self, num: int) -> bool:
        def rev(n: int) -> int:
            r = 0
            while n > 0:
                r = r * 10 + n % 10
                n //= 10
            return r
        return rev(rev(num)) == num

print(SolutionMath().isSameAfterReversals(1800))

False


Độ phức tạp: **Time `O(d)`**, **Space `O(1)`**.

### Cách 3 — Quy tắc `num % 10` (một dòng)

Dùng kết luận ở mục 6: `return num == 0 or num % 10 != 0`.

In [11]:
def same_rule(num):
    return num == 0 or num % 10 != 0

for num, expected in examples:
    print(num, same_rule(num), expected)

526 True True
1800 False False
0 True True


In [12]:
class SolutionRule:
    def isSameAfterReversals(self, num: int) -> bool:
        return num == 0 or num % 10 != 0

print(SolutionRule().isSameAfterReversals(526))

True


Độ phức tạp: **Time `O(1)`**, **Space `O(1)`**.

### Cách 4 — Quy tắc trên chuỗi

Cùng ý tưởng nhưng đọc chữ số cuối từ chuỗi: `s = str(num)`; đúng khi `s == "0"` hoặc `s[-1] != "0"`. Hữu ích khi `num` được cho dưới dạng chuỗi rất dài.

In [13]:
def same_rule_str(num):
    s = str(num)
    return s == "0" or s[-1] != "0"

for num, expected in examples:
    print(num, same_rule_str(num), expected)

526 True True
1800 False False
0 True True


In [14]:
class SolutionStrRule:
    def isSameAfterReversals(self, num: int) -> bool:
        s = str(num)
        return s == "0" or not s.endswith("0")

print(SolutionStrRule().isSameAfterReversals(10))

False


Độ phức tạp: **Time `O(d)`** (chuyển sang chuỗi), **Space `O(d)`**.

## 8. TRACE từng bước

Trace Cách 2 với `num = 1800`:

| lần đảo | n trước | vòng | chữ số `n%10` | rev | n sau |
|---|---|---|---|---|---|
| 1 | 1800 | 1 | 0 | 0 | 180 |
| 1 | 180 | 2 | 0 | 0 | 18 |
| 1 | 18 | 3 | 8 | 8 | 1 |
| 1 | 1 | 4 | 1 | 81 | 0 |
| 2 | 81 | 1 | 1 | 1 | 8 |
| 2 | 8 | 2 | 8 | 18 | 0 |

Hai chữ số 0 đầu bị "nuốt" vì `rev = 0*10 + 0 = 0`: ghép 0 vào số 0 vẫn là 0.

In [15]:
def trace_reverse(n):
    print("đảo", n)
    rev = 0
    step = 0
    while n > 0:
        step += 1
        digit = n % 10
        rev = rev * 10 + digit
        n //= 10
        print(f"  vòng {step}: digit={digit}, rev={rev}, n còn lại={n}")
    print("  => kết quả", rev)
    return rev

r1 = trace_reverse(1800)
r2 = trace_reverse(r1)
print("reversed2 == num ?", r2 == 1800)

đảo 1800
  vòng 1: digit=0, rev=0, n còn lại=180
  vòng 2: digit=0, rev=0, n còn lại=18
  vòng 3: digit=8, rev=8, n còn lại=1
  vòng 4: digit=1, rev=81, n còn lại=0
  => kết quả 81
đảo 81
  vòng 1: digit=1, rev=1, n còn lại=8
  vòng 2: digit=8, rev=18, n còn lại=0
  => kết quả 18
reversed2 == num ? False


In [16]:
def trace_rule(num):
    last = num % 10
    print(f"num={num}, chữ số cuối={last}, num==0? {num == 0}")
    ans = num == 0 or last != 0
    print("kết quả:", ans)
    return ans

for n in [526, 1800, 0, 10]:
    trace_rule(n)

num=526, chữ số cuối=6, num==0? False
kết quả: True
num=1800, chữ số cuối=0, num==0? False
kết quả: False
num=0, chữ số cuối=0, num==0? True
kết quả: True
num=10, chữ số cuối=0, num==0? False
kết quả: False


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def isSameAfterReversals(self, num: int) -> bool:
        return num == 0 or num % 10 != 0

print(Solution().isSameAfterReversals(526), Solution().isSameAfterReversals(1800), Solution().isSameAfterReversals(0))

True False True


## 10. Vì sao thuật toán đúng?

**Khi `num % 10 != 0`.** Viết `num` có `k` chữ số `d₁...dₖ` với `dₖ ≠ 0`. Đảo lần 1 cho dãy `dₖ...d₁` bắt đầu bằng `dₖ ≠ 0` nên không bị cắt bớt, đúng `k` chữ số. Đảo lần 2 trả về `d₁...dₖ` — chính `num`. Vậy `True`.

**Khi `num % 10 == 0` và `num > 0`.** Giả sử `num` có `z ≥ 1` chữ số 0 ở cuối. Dãy đảo lần 1 có `z` chữ số 0 ở đầu, bị bỏ, nên `reversed1` chỉ còn `k - z` chữ số. Đảo lần 2 không thể tăng số chữ số, nên `reversed2` có tối đa `k - z < k` chữ số và `reversed2 < 10^(k-z) ≤ num`... cụ thể `reversed2 ≠ num`. Vậy `False`.

**Khi `num = 0`.** `0 → 0 → 0`, `True`.

Ta cũng sẽ kiểm chứng bằng brute force cho **mọi** `0 ≤ num ≤ 10^6` ở mục bộ test.

### Chứng minh bằng đếm chữ số (gọn hơn)

Gọi `len(x)` là số chữ số của `x` (với `x > 0`). Phép đảo không bao giờ tăng số chữ số: `len(reverse(x)) ≤ len(x)`, và bằng nhau **khi và chỉ khi** chữ số cuối của `x` khác 0. Nếu `num` tận cùng bằng 0 thì `len(reversed1) < len(num)`, suy ra `len(reversed2) ≤ len(reversed1) < len(num)`, nên `reversed2 ≠ num`. Ngược lại, nếu không tận cùng bằng 0 thì cả hai lần đảo đều giữ nguyên số chữ số, và hai lần đảo thứ tự chữ số trở về đúng thứ tự ban đầu.

In [18]:
# kiểm chứng tính chất len(reverse(x)) <= len(x) và dấu bằng khi chữ số cuối khác 0
for x in range(1, 100001):
    L, L1 = len(str(x)), len(str(reverse_math(x)))
    assert L1 <= L
    assert (L1 == L) == (x % 10 != 0)
print("len(reverse(x)) <= len(x), bằng nhau <=> x % 10 != 0: đúng với 1..100000")

len(reverse(x)) <= len(x), bằng nhau <=> x % 10 != 0: đúng với 1..100000


### Bảng tổng hợp theo chữ số cuối

| chữ số cuối của `num` | ví dụ | `reversed1` | `reversed2` | kết quả |
|---|---|---|---|---|
| khác 0 | `526` | `625` | `526` | `True` |
| khác 0, có 0 ở giữa | `1005` | `5001` | `1005` | `True` |
| bằng 0 (`num > 0`) | `1800` | `81` | `18` | `False` |
| bằng 0 duy nhất | `0` | `0` | `0` | `True` |

Nhìn bảng, ta thấy kết quả chỉ phụ thuộc cột đầu tiên: **chữ số cuối**. Hãy tự thêm vài số của riêng bạn vào bảng trước khi chạy code ở dưới để xác nhận.

In [19]:
rows = [526, 1005, 1800, 0, 90, 9, 1000000]
print(f"{'num':>8} {'rev1':>8} {'rev2':>8} {'giống?':>7} {'cuối':>5}")
for num in rows:
    r1 = reverse_math(num)
    r2 = reverse_math(r1)
    print(f"{num:>8} {r1:>8} {r2:>8} {str(r2 == num):>7} {num % 10:>5}")

     num     rev1     rev2  giống?  cuối
     526      625      526    True     6
    1005     5001     1005    True     5
    1800       81       18   False     0
       0        0        0    True     0
      90        9        9   False     0
       9        9        9    True     9
 1000000        1        1   False     0


### Ghi chú thêm: vì sao đề cho `num ≤ 10^6` mà quy tắc vẫn đúng với số rất lớn?

Chứng minh ở trên không dùng giới hạn của `num`, nên quy tắc `num == 0 or num % 10 != 0` đúng với **mọi** số nguyên không âm. Giới hạn `10^6` chỉ giúp ta *kiểm chứng bằng brute force* được trong vài giây. Trong Python, số nguyên có độ chính xác tùy ý nên cả mô phỏng lẫn quy tắc đều chạy được với số có hàng trăm chữ số; còn ở ngôn ngữ có số nguyên 32-bit (C++, Java), việc đảo số bằng toán có thể **tràn số**, trong khi quy tắc `% 10` thì không bao giờ tràn. Đó là một lý do thực tế nữa để thích quy tắc ngắn.

In [20]:
big = 10 ** 60 + 7          # số rất lớn, chữ số cuối là 7
print(same_rule(big), same_simulate_math(big))
big2 = 7 * 10 ** 60         # tận cùng bằng 60 chữ số 0
print(same_rule(big2), same_simulate_math(big2))

True True
False False


## 11. Phân tích Time / Space Complexity

Gọi `d` là số chữ số của `num` (`d ≤ 7` vì `num ≤ 10^6`).

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng chuỗi | `O(d)` | `O(d)` |
| 2. Mô phỏng toán | `O(d)` | `O(1)` |
| 3. Quy tắc `% 10` | `O(1)` | `O(1)` |
| 4. Quy tắc chuỗi | `O(d)` | `O(d)` |

Không có sự khác biệt worst/average: mỗi cách cùng chi phí với mọi input cùng số chữ số. Cách 3 là tốt nhất nhưng Cách 1–2 giúp bạn **kiểm chứng** quy tắc.

## 12. Bộ test

Vì miền input nhỏ (`0..10^6`), ta kiểm chứng quy tắc `O(1)` trên **toàn bộ** miền so với mô phỏng, sau đó test thêm edge case và input ngẫu nhiên.

In [21]:
cases = [(526, True), (1800, False), (0, True), (1, True), (10, False), (100, False),
         (101, True), (1005, True), (1050, False), (999999, True), (1000000, False), (120, False)]
impls = [same_simulate_str, same_simulate_math, same_rule, same_rule_str,
         Solution().isSameAfterReversals, SolutionMath().isSameAfterReversals,
         SolutionRule().isSameAfterReversals, SolutionStrRule().isSameAfterReversals]
for num, expected in cases:
    for f in impls:
        assert f(num) == expected, (f.__name__, num)
print("Tất cả", len(cases), "ca x", len(impls), "cách đều đúng")

Tất cả 12 ca x 8 cách đều đúng


In [22]:
# kiểm chứng quy tắc trên TOÀN BỘ miền 0..10^6 so với mô phỏng chuỗi
for num in range(0, 1_000_001):
    assert same_rule(num) == same_simulate_str(num), num
print("quy tắc num == 0 or num % 10 != 0 đúng với mọi num trong [0, 10^6]")

quy tắc num == 0 or num % 10 != 0 đúng với mọi num trong [0, 10^6]


In [23]:
import random
random.seed(2119)
for _ in range(5000):
    num = random.randint(0, 10**6)
    want = same_simulate_math(num)
    assert same_simulate_str(num) == want
    assert same_rule_str(num) == want
    assert Solution().isSameAfterReversals(num) == want
print("5000 input ngẫu nhiên: mọi cách khớp nhau")

5000 input ngẫu nhiên: mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên trường hợp `num = 0`

`0 % 10 == 0`, nên công thức `num % 10 != 0` một mình trả `False` cho `0`, nhưng đáp án đúng là `True`.

In [24]:
def wrong1(num):
    return num % 10 != 0       # SAI cho num = 0

print("sai:", wrong1(0), "| đúng:", same_rule(0))

sai: False | đúng: True


### Lỗi 2: Kiểm tra "có chữ số 0" thay vì "tận cùng bằng 0"

`1005` có chữ số 0 nhưng đảo hai lần vẫn là `1005` (`True`). Chỉ số 0 ở **cuối** mới gây mất mát.

In [25]:
def wrong2(num):
    return "0" not in str(num) or num == 0     # SAI: 1005 bị coi là False

print("sai:", wrong2(1005), "| đúng:", same_rule(1005))

sai: False | đúng: True


### Lỗi 3: Đảo số bằng toán nhưng điều kiện `while n >= 0`

Với `n >= 0`, `n` không bao giờ âm (vì `0 // 10 = 0`), nên vòng lặp **chạy mãi**. Điều kiện phải là `n > 0`.

In [26]:
def reverse_wrong(n):
    rev = 0
    steps = 0
    while n >= 0 and steps < 5:      # có giới hạn 5 vòng chỉ để minh họa lỗi (thật sự lặp vô hạn)
        rev = rev * 10 + n % 10
        n //= 10
        steps += 1
    return rev, steps

print(reverse_wrong(12))
print("đúng:", reverse_math(12))

(21000, 5)
đúng: 21


### Lỗi 4: So sánh `int` với chuỗi

`str(reversed2) == num` hoặc `reversed2 == str(num)` luôn `False` vì khác kiểu. Hãy so sánh cùng kiểu.

In [27]:
num = 526
r2 = int(str(int(str(num)[::-1]))[::-1])
print(r2 == str(num), "<- sai kiểu")
print(r2 == num, "<- đúng")

False <- sai kiểu
True <- đúng


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "làm một phép biến đổi hai lần rồi so với ban đầu" thường chỉ hỏi **thông tin nào bị mất** trong phép biến đổi. Hãy tìm quy tắc ngắn thay vì mô phỏng.
- Khi miền input nhỏ, hãy **brute force mọi giá trị** để kiểm chứng quy tắc bạn đoán (đúng như ta làm ở mục 12).
- Nhớ edge case `0`, số một chữ số, số có nhiều chữ số 0 cuối.
- `n % 10` lấy chữ số cuối; `n // 10` bỏ chữ số cuối: hai công cụ nền cho mọi bài về chữ số (Palindrome Number, Reverse Integer, Add Digits...).
- Đừng đảo bằng chuỗi nếu đề cấm chuyển số sang chuỗi — hãy dùng Cách 2.

## 15. Biến thể của bài

1. Đảo **một lần** và hỏi có bằng `num` không (đó chính là bài Palindrome Number).
2. Số `num` có thể **âm** (khi đó dấu cũng đảo hay giữ?).
3. `num` rất lớn cho dưới dạng **chuỗi**: Cách 4 vẫn chạy.
4. Đếm có bao nhiêu `num` trong `[0, N]` thỏa điều kiện: đáp án là `1 + (số không chia hết cho 10 trong [1, N])`.
5. Đảo ngược theo **hệ cơ số khác** (ví dụ nhị phân): "tận cùng bằng 0" đổi thành "chẵn".

In [28]:
def count_true(N):
    return sum(1 for x in range(N + 1) if same_rule(x))

def count_true_formula(N):
    return 1 + (N - N // 10)       # 0 luôn đúng; trong 1..N có N//10 số chia hết cho 10

for N in [0, 9, 10, 99, 100, 1000]:
    print(N, count_true(N), count_true_formula(N))
    assert count_true(N) == count_true_formula(N)

0 1 1
9 10 10
10 10 10
99 91 91
100 91 91
1000 901 901


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng chuỗi | `int(str(n)[::-1])` hai lần | `O(d)` | `O(d)` | dễ hiểu, kiểm chứng |
| 2. Mô phỏng toán | `%10`, `//10` | `O(d)` | `O(1)` | khi cấm dùng chuỗi |
| 3. Quy tắc `% 10` | 0 hoặc không tận cùng 0 | `O(1)` | `O(1)` | nộp bài, tối ưu nhất |
| 4. Quy tắc chuỗi | `s[-1] != "0"` | `O(d)` | `O(d)` | số cho dạng chuỗi |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `same_rule(2050)` và `same_rule(5020)` trả về gì?

**Bài 2 (tính tay):** đảo hai lần số `4300`. Ghi `reversed1`, `reversed2`.

**Bài 3 (điền TODO):** hoàn thành `exercise_reverse(n)` bằng `%` và `//`.

**Bài 4 (tự viết lại):** viết `exercise_rule(num)` trong một dòng.

**Bài 5 (mini challenge):** viết `exercise_count(N)` đếm số `x` trong `[0, N]` thỏa điều kiện, **không** dùng vòng lặp (dùng công thức).

In [29]:
# Bài 1: chạy để so với dự đoán
print(same_rule(2050), same_rule(5020))

False False


In [30]:
def exercise_reverse(n):
    rev = 0
    # TODO: while n > 0: rev = rev * 10 + n % 10; n //= 10
    # TODO: return rev
    pass

def exercise_rule(num):
    # TODO: một dòng với num == 0 hoặc chữ số cuối khác 0
    pass

def exercise_count(N):
    # TODO: 1 + (N - N // 10)
    pass

# (không gọi khi chưa làm xong)

## 18. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Nhìn chữ số cuối | `2050` cuối là 0, `5020` cuối là 0 | cả hai `False` |
| 2 | `4300` → bỏ hai số 0 đầu sau khi đảo | `reversed1 = 34` | `reversed2 = 43 ≠ 4300` |
| 3 | `n % 10` lấy chữ số cuối | `rev * 10 + digit` | `n //= 10` mỗi vòng |
| 4 | Hai trường hợp đúng: `0` hoặc không tận cùng 0 | dùng `or` | `return num == 0 or num % 10 != 0` |
| 5 | Có `N // 10` số chia hết cho 10 trong `1..N` | Đếm số *không* chia hết rồi cộng số `0` | `1 + N - N // 10` |

## 19. Đáp án tham khảo cho bài tập

Bài 1: `False False`. Bài 2: `reversed1 = 34`, `reversed2 = 43`, khác `4300`.

In [31]:
def exercise_reverse(n):
    rev = 0
    while n > 0:
        rev = rev * 10 + n % 10
        n //= 10
    return rev

def exercise_rule(num):
    return num == 0 or num % 10 != 0

def exercise_count(N):
    return 1 + N - N // 10

assert same_rule(2050) is False and same_rule(5020) is False
assert exercise_reverse(4300) == 34 and exercise_reverse(34) == 43
assert exercise_reverse(0) == 0
assert exercise_rule(0) and exercise_rule(7) and not exercise_rule(30)
assert all(exercise_rule(x) == same_simulate_str(x) for x in range(2000))
assert exercise_count(0) == 1 and exercise_count(10) == 10 and exercise_count(100) == 91
assert all(exercise_count(N) == count_true(N) for N in range(300))
print("Đáp án bài tập đều đúng")

Đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
return num == 0 or num % 10 != 0
```

**Checklist:**

- [ ] Đã xử lý riêng `num = 0` (đáp án `True`)?
- [ ] Hiểu rằng chỉ số 0 ở **cuối** mới làm mất thông tin?
- [ ] Nếu mô phỏng bằng toán, điều kiện vòng lặp là `n > 0`?
- [ ] So sánh cùng kiểu (`int` với `int`)?
- [ ] Đã kiểm chứng quy tắc bằng brute force trên miền nhỏ?